# 03 — EDA Avanzado y Validación de Hipótesis

**Objetivo:** Confirmar los patrones clave antes de configurar los modelos Prophet (Fase 4).

**Fuente de datos:** `data/processed/panel_unificado.csv` (Reglas 1 y 2 ya aplicadas).

**Análisis que se realizan:**

| Análisis | Hipótesis a validar |
|---|---|
| **A** — Estacionalidad por sede (Prórroga) | ¿El pico de enero es solo un efecto de Lima o se repite en toda la red? |
| **B** — Test de Granger (Visas → Cambio) | ¿Existe relación líder-rezago estadísticamente significativa entre solicitud_visas y cambio_calidad? |
| **C** — Quiebre estructural visual | ¿El quiebre dic-2025→ene-2026 es real y simétrico, o parcialmente un artefacto de rezago de reporte? |

> 📌 Este notebook **no entrena modelos** — eso es Fase 4.

In [ ]:
import sys
import warnings
import io
import contextlib
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUTA_PANEL = ROOT / "data" / "processed" / "panel_unificado.csv"
panel = pd.read_csv(RUTA_PANEL, parse_dates=["ds"])

print(f"Panel cargado: {len(panel):,} filas | {panel['ds'].min().date()} a {panel['ds'].max().date()}")
print(f"Tipos de tramite: {panel['tipo_tramite'].unique().tolist()}")
print(f"Sedes: {panel['sede'].nunique()} unicas")

---
## Análisis A — Estacionalidad de Prórroga de Residencia por sede

**Hipótesis:** El pico de enero visible a nivel nacional es solo un efecto arrastrado por Lima
(que concentra ~78% del volumen), y las sedes regionales no muestran ese patrón.

**Criterio de confirmación:** Si el ratio enero/promedio en sedes regionales es > 1.3x,
el patrón es sistémico (no solo Lima).

In [ ]:
prorroga = panel[panel["tipo_tramite"] == "prorroga_residencia"].copy()

top5 = (
    prorroga.groupby("sede")["y"].sum()
    .nlargest(5)
    .reset_index()
    .rename(columns={"y": "total"})
)
top5["pct"] = top5["total"] / prorroga["y"].sum() * 100
top5_list = top5["sede"].tolist()

print("Top 5 sedes — Prorroga de Residencia (2025-2026):")
print(top5.assign(total=top5["total"].apply(lambda x: f"{int(x):,}"),
                  pct=top5["pct"].apply(lambda x: f"{x:.1f}%")).to_string(index=False))

In [ ]:
ORDEN_MESES = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
               7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

prorroga["mes"] = prorroga["ds"].dt.month
pivot_mes = (
    prorroga[prorroga["sede"].isin(top5_list)]
    .groupby(["sede", "mes"])["y"].sum()
    .reset_index()
)

print("Patron estacional por sede (suma 2025+2026):")
print("-" * 75)
resumen_eda = []
for sede in top5_list:
    df_s = pivot_mes[pivot_mes["sede"] == sede].set_index("mes")["y"]
    mes_max = df_s.idxmax()
    mes_min = df_s.idxmin()
    ratio = df_s[mes_max] / df_s[mes_min]
    ene = df_s.get(1, 0)
    prom = df_s.mean()
    ratio_ene = ene / prom if prom > 0 else 0
    resumen_eda.append({
        "sede": sede, "mes_pico": ORDEN_MESES[mes_max], "mes_valle": ORDEN_MESES[mes_min],
        "ratio_pico/valle": round(ratio, 1), "ratio_ene/prom": round(ratio_ene, 2)
    })
    print(f"  {sede:<12} pico={ORDEN_MESES[mes_max]}  valle={ORDEN_MESES[mes_min]}  "
          f"ratio_pico/valle={ratio:.1f}x  Ene/Promedio={ratio_ene:.2f}x")

print()
todas_con_pico_ene = all(r["mes_pico"] == "Ene" for r in resumen_eda)
todas_sobre_umbral = all(r["ratio_ene/prom"] > 1.3 for r in resumen_eda)
print(f"Todas con pico en enero:       {'SI' if todas_con_pico_ene else 'NO'}")
print(f"Todas con ratio Ene/Prom>1.3x: {'SI' if todas_sobre_umbral else 'NO'}")

In [ ]:
prorroga_2025 = prorroga[prorroga["ds"].dt.year == 2025]
pivot_serie = (
    prorroga_2025[prorroga_2025["sede"].isin(top5_list)]
    .groupby(["ds", "sede"])["y"].sum()
    .reset_index()
)

COLORES_SEDE = {
    "LIMA": "#1565C0", "TRUJILLO": "#E65100", "AREQUIPA": "#2E7D32",
    "PIURA": "#6A1B9A", "CHICLAYO": "#00695C"
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for sede in top5_list:
    sub = pivot_serie[pivot_serie["sede"] == sede].sort_values("ds")
    ax.plot(sub["ds"], sub["y"], marker="o", label=sede, color=COLORES_SEDE.get(sede, "gray"), linewidth=2)
ax.set_title("Prorroga de Residencia — Serie mensual por sede (2025)", fontsize=10)
ax.set_ylabel("Cantidad")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax2 = axes[1]
for sede in top5_list:
    sub = pivot_serie[pivot_serie["sede"] == sede].sort_values("ds")
    base = sub["y"].iloc[0]
    if base > 0:
        ax2.plot(sub["ds"], sub["y"] / base * 100, marker="o",
                 label=sede, color=COLORES_SEDE.get(sede, "gray"), linewidth=2)
ax2.axhline(100, color="gray", linestyle=":", alpha=0.5)
ax2.set_title("Misma serie indexada (Ene-2025 = 100)\nMisma forma en todas las sedes", fontsize=10)
ax2.set_ylabel("Indice (Ene = 100)")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("CONCLUSION — Analisis A:")
print("  El patron estacional (pico Ene, valle Sep) se repite en TODAS las sedes del top-5.")
print("  La diferencia es de intensidad (Lima: 1.90x; Chiclayo: 1.41x), no de forma.")
print("  -> yearly_seasonality=True confirmado para Prorroga de Residencia.")

---
## Análisis B — Test de Causalidad de Granger

**Hipótesis:** `solicitud_visas` actúa como indicador líder de `cambio_calidad`,
con un rezago de 1-2 meses.

**Metodología:**
1. Verificar estacionariedad (ADF) en niveles y primeras diferencias.
2. Aplicar test de Granger en ambas direcciones para lags 1 y 2.
3. Interpretar con cautela: muestra de solo 19 obs. tras diferenciación.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests, adfuller

def serie_nac(tipo):
    return (
        panel[panel["tipo_tramite"] == tipo]
        .groupby("ds")["y"].sum()
        .sort_index()
    )

sv = serie_nac("solicitud_visas").rename("solicitud_visas")
cc = serie_nac("cambio_calidad").rename("cambio_calidad")
series = pd.concat([sv, cc], axis=1).sort_index()

print(f"Series: {len(series)} meses")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(series.index, series["solicitud_visas"], color="#9C27B0", marker="o", linewidth=2)
ax1.set_title("Solicitud de Calidad Migratoria (Visas) — nacional", fontsize=10)
ax1.set_ylabel("Cantidad")
ax1.grid(alpha=0.3)
ax2.plot(series.index, series["cambio_calidad"], color="#4CAF50", marker="o", linewidth=2)
ax2.set_title("Cambio de Calidad Migratoria — nacional", fontsize=10)
ax2.set_ylabel("Cantidad")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("PASO 1 — Test ADF en niveles:")
for col in ["solicitud_visas", "cambio_calidad"]:
    r = adfuller(series[col], autolag="AIC")
    print(f"  {col:<28}  p={r[1]:.4f}  {'ESTACIONARIA' if r[1] < 0.05 else 'NO estacionaria'}")

series_diff = series.diff().dropna()
print()
print("PASO 2 — Test ADF en primeras diferencias:")
for col in ["solicitud_visas", "cambio_calidad"]:
    r = adfuller(series_diff[col], autolag="AIC")
    print(f"  d({col:<26})  p={r[1]:.4f}  {'ESTACIONARIA' if r[1] < 0.05 else 'marginal/NO'}")

def run_granger(data):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        res = grangercausalitytests(data, maxlag=2)
    rows = []
    for lag in [1, 2]:
        pf  = res[lag][0]["ssr_ftest"][1]
        pc2 = res[lag][0]["ssr_chi2test"][1]
        rows.append({"lag": lag, "F_p": round(pf, 4), "chi2_p": round(pc2, 4),
                     "significativo": pf < 0.05})
    return pd.DataFrame(rows)

print()
print("PASO 3 — Test de Granger (primeras diferencias):")
print()
print("Test A: solicitud_visas CAUSA (Granger) cambio_calidad?")
df_a = run_granger(series_diff[["cambio_calidad", "solicitud_visas"]].values)
for _, row in df_a.iterrows():
    sig = "SIGNIFICATIVO (p<0.05)" if row["significativo"] else "no significativo"
    print(f"  Lag {int(row['lag'])}:  F-test p={row['F_p']:.4f}  chi2 p={row['chi2_p']:.4f}  -> {sig}")

print()
print("Test B: cambio_calidad CAUSA (Granger) solicitud_visas?")
df_b = run_granger(series_diff[["solicitud_visas", "cambio_calidad"]].values)
for _, row in df_b.iterrows():
    sig = "SIGNIFICATIVO (p<0.05)" if row["significativo"] else "no significativo"
    print(f"  Lag {int(row['lag'])}:  F-test p={row['F_p']:.4f}  chi2 p={row['chi2_p']:.4f}  -> {sig}")

print()
print("CONCLUSION — Analisis B:")
print("  Lag 2 del Test A es significativo (p=0.024), pero muestra = 19 obs. (baja potencia).")
print("  d(solicitud_visas) marginal (ADF p=0.052). Lag 1 no es significativo.")
print("  DECISION: no usar como regresor en Fase 4. Revisar con >= 24 meses.")

---
## Análisis C — Quiebre estructural: verificación con columna `regimen`

In [ ]:
TIPO_CONFIG = {
    "cambio_calidad":      {"color": "#4CAF50", "tiene_quiebre": True,  "label": "Cambio de Calidad Migratoria"},
    "carnet_extranjeria":  {"color": "#FF9800", "tiene_quiebre": True,  "label": "Carne de Extranjeria (neto)"},
    "prorroga_residencia": {"color": "#2196F3", "tiene_quiebre": False, "label": "Prorroga de Residencia"},
    "solicitud_visas":     {"color": "#9C27B0", "tiene_quiebre": False, "label": "Solicitud de Calidad (Visas)"},
}

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=False)
axes_flat = axes.flatten()

for ax, (tipo, cfg) in zip(axes_flat, TIPO_CONFIG.items()):
    serie = (
        panel[panel["tipo_tramite"] == tipo]
        .groupby("ds")["y"].sum()
        .sort_index()
        .reset_index()
    )
    if cfg["tiene_quiebre"]:
        pre  = serie[serie["ds"].dt.year <= 2025]
        post = serie[serie["ds"].dt.year >= 2026]
        ax.plot(pre["ds"],  pre["y"],  color=cfg["color"], linewidth=2.5, marker="o", label="pre_2026")
        ax.plot(post["ds"], post["y"], color=cfg["color"], linewidth=2.5, marker="s",
                linestyle="--", alpha=0.75, label="post_2026")
        ax.axvline(pd.Timestamp("2026-01-01"), color="red", linestyle="--", linewidth=1.5, alpha=0.8)
        prom_pre  = pre["y"].mean()
        prom_post = post["y"].mean()
        ax.axhline(prom_pre,  color=cfg["color"], linestyle=":", alpha=0.5)
        ax.axhline(prom_post, color=cfg["color"], linestyle=":", alpha=0.5)
        caida = (prom_post - prom_pre) / prom_pre * 100
        ax.set_title(f"{cfg['label']}\nprom_pre={prom_pre:,.0f} → prom_post={prom_post:,.0f} ({caida:+.1f}%)", fontsize=9)
    else:
        ax.plot(serie["ds"], serie["y"], color=cfg["color"], linewidth=2.5, marker="o")
        ax.axvline(pd.Timestamp("2026-01-01"), color="gray", linestyle="--", linewidth=1, alpha=0.5)
        ax.set_title(f"{cfg['label']}\n(sin quiebre estructural)", fontsize=9)
    ax.set_ylabel("Cantidad")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle("Analisis C — Quiebre estructural dic-2025 a ene-2026", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Análisis C.2 — ¿El 'quiebre' de carnet_extranjeria es real o un artefacto de rezago de reporte?

**Hipótesis adicional:** Los trámites de Carné se registran semanas/meses *después* del Cambio
de Calidad que los origina. Por tanto, los últimos 1-2 meses del panel (en particular
agosto 2026) pueden estar **subreportados** — no reflejan una caída real de demanda sino
el retraso en la carga de datos.

**Criterio:** Si al excluir el último mes el promedio 2026 sube y se acerca al de 2025,
el quiebre es parcialmente artefactual. Comparar con `cambio_calidad` (donde el quiebre
sí es real) como control.

In [ ]:
carnet_serie = (
    panel[panel["tipo_tramite"] == "carnet_extranjeria"]
    .groupby("ds")["y"].sum()
    .sort_index()
    .reset_index()
)
carnet_serie["anio"] = carnet_serie["ds"].dt.year

prom_2025 = carnet_serie.loc[carnet_serie["anio"] == 2025, "y"].mean()
datos_2026 = carnet_serie[carnet_serie["anio"] == 2026].copy().reset_index(drop=True)

prom_8m = datos_2026["y"].mean()               # ene-ago (todos)
prom_7m = datos_2026.iloc[:-1]["y"].mean()     # ene-jul (excl. ago)
prom_6m = datos_2026.iloc[:-2]["y"].mean()     # ene-jun (excl. jul+ago)

print("Serie mensual carnet_extranjeria 2026 (variacion mes a mes):")
print("-" * 50)
prev = None
for _, row in datos_2026.iterrows():
    lbl = row["ds"].strftime("%b-%Y")
    var = f"  ({(row['y']-prev)/prev*100:+.1f}% vs anterior)" if prev is not None else "  (base)"
    print(f"  {lbl:<10}  {int(row['y']):>7,}{var}")
    prev = row["y"]

print()
print(f"Promedio 2025 (12 meses)          : {prom_2025:>7,.1f}")
print(f"Promedio 2026 — 8 meses (ene-ago) : {prom_8m:>7,.1f}  ({(prom_8m-prom_2025)/prom_2025*100:+.1f}% vs 2025)")
print(f"Promedio 2026 — 7 meses (ene-jul) : {prom_7m:>7,.1f}  ({(prom_7m-prom_2025)/prom_2025*100:+.1f}% vs 2025)")
print(f"Promedio 2026 — 6 meses (ene-jun) : {prom_6m:>7,.1f}  ({(prom_6m-prom_2025)/prom_2025*100:+.1f}% vs 2025)")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

pre  = carnet_serie[carnet_serie["anio"] == 2025]
post = carnet_serie[carnet_serie["anio"] == 2026]

ax.plot(pre["ds"],  pre["y"],  color="#FF9800", linewidth=2.5, marker="o", label="2025")
ax.plot(post["ds"], post["y"], color="#FF9800", linewidth=2.5, marker="s",
        linestyle="--", alpha=0.8, label="2026")

ax.axhline(prom_2025, color="blue",   linestyle=":", linewidth=1.5,
           label=f"Prom 2025 (12m): {prom_2025:,.0f}")
ax.axhline(prom_8m,   color="red",    linestyle=":", linewidth=1.5,
           label=f"Prom 2026 (8m, ene-ago): {prom_8m:,.0f}  ({(prom_8m-prom_2025)/prom_2025*100:+.1f}%)")
ax.axhline(prom_7m,   color="green",  linestyle=":", linewidth=1.5,
           label=f"Prom 2026 (7m, ene-jul): {prom_7m:,.0f}  ({(prom_7m-prom_2025)/prom_2025*100:+.1f}%)")
ax.axhline(prom_6m,   color="purple", linestyle=":", linewidth=1.5,
           label=f"Prom 2026 (6m, ene-jun): {prom_6m:,.0f}  ({(prom_6m-prom_2025)/prom_2025*100:+.1f}%)")

ax.axvline(pd.Timestamp("2026-01-01"), color="red", linestyle="--",
           linewidth=1.5, alpha=0.5, label="Ene 2026")

# Destacar agosto 2026 (posible subreporte)
ago = post[post["ds"] == "2026-08-01"]
if not ago.empty:
    ax.scatter(ago["ds"], ago["y"], color="red", s=140, zorder=5,
               label=f"Ago-2026: {int(ago['y'].iloc[0]):,} (posible subreporte)")

ax.set_title("Carne de Extranjeria — Analisis de rezago de reporte\n"
             "Comparacion de promedios con distintos recortes del ultimo mes", fontsize=10)
ax.set_ylabel("Cantidad")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Control: ¿cambio_calidad muestra la misma caída en agosto 2026?
cc_26 = (
    panel[panel["tipo_tramite"] == "cambio_calidad"]
    .groupby("ds")["y"].sum()
    .sort_index()
)
prom_cc_2025 = cc_26[cc_26.index.year == 2025].mean()
datos_cc_26  = cc_26[cc_26.index.year == 2026]
prom_cc_8m   = datos_cc_26.mean()
prom_cc_7m   = datos_cc_26.iloc[:-1].mean()

print("REFERENCIA — cambio_calidad 2026 (quiebre real):")
print(f"  Prom 2025 (12m): {prom_cc_2025:>7,.1f}")
print(f"  Prom 2026 (8m) : {prom_cc_8m:>7,.1f}  ({(prom_cc_8m-prom_cc_2025)/prom_cc_2025*100:+.1f}%)")
print(f"  Prom 2026 (7m) : {prom_cc_7m:>7,.1f}  ({(prom_cc_7m-prom_cc_2025)/prom_cc_2025*100:+.1f}%)")
print()
print("En cambio_calidad, excluir agosto NO cambia el cuadro (-38.5% -> -38.3%).")
print("El quiebre en cambio_calidad es ESTRUCTURAL, no artefacto de reporte.")
print()
print("En carnet_extranjeria, excluir agosto SI cambia el promedio:")
print(f"  Con 8m: {prom_8m:,.1f} ({(prom_8m-prom_2025)/prom_2025*100:+.1f}%)")
print(f"  Con 7m: {prom_7m:,.1f} ({(prom_7m-prom_2025)/prom_2025*100:+.1f}%)  <- excl. ago")
print(f"  Con 6m: {prom_6m:,.1f} ({(prom_6m-prom_2025)/prom_2025*100:+.1f}%)  <- excl. jul+ago")
print()
print("Ago-2026 cae -15.9% vs jul-2026 en carnet, mientras cambio_calidad en el")
print("mismo mes es estable (-6.8%). Esto confirma que la caida de ago es SUBREPORTE.")

In [ ]:
print("CONCLUSION FINAL — Analisis C completo:")
print()
print("  cambio_calidad:")
print("    Quiebre SEVERO y REAL: -51.7% puntual en ene-2026, -38.5% de nivel.")
print("    Excluir ago-2026 no cambia el cuadro (-38.3%). No es rezago.")
print("    DECISION: solo post_2026 (8 meses). yearly_seasonality=False.")
print()
print("  carnet_extranjeria:")
print("    Quiebre DEBIL y PARCIALMENTE ARTEFACTUAL:")
print(f"      8 meses (ene-ago): prom={prom_8m:,.0f}  (-8.0% vs 2025)")
print(f"      7 meses (ene-jul): prom={prom_7m:,.0f}  (-5.3% vs 2025)")
print(f"      6 meses (ene-jun): prom={prom_6m:,.0f}  (-4.0% vs 2025)")
print("      Ago-2026 cae -15.9% vs jul, pero cambio_calidad en el mismo mes es estable.")
print("      Conclusion: la caida de ago es subreporte, no demanda real.")
print()
print("    DECISION REVISADA (ver docs/decisiones_modelado.md):")
print("      Usar los 19 meses (ene-2025 a jul-2026, excl. ago-2026 por subreporte).")
print("      NO separar por regimen: el quiebre no supera el umbral de ruido de reporte.")
print("      yearly_seasonality: EVALUAR en Fase 4 (True vs False por validacion cruzada).")

---
## Resumen de decisiones de la Fase 3

| Análisis | Resultado | Decisión para Fase 4 |
|---|---|---|
| **A** — Estacionalidad Prórroga | Patrón enero sistémico en **todas** las sedes (ratios 1.41–1.90x) | `yearly_seasonality=True`, modelo por sede |
| **B** — Granger Visas→Cambio | Significativo en lag 2 (p=0.024) pero muestra pequeña (19 obs.) | No usar como regresor; revisar con ≥ 24 meses |
| **C** — Quiebre estructural | Severo y real en `cambio_calidad` (−51.7%); débil/artefactual en `carnet` (−4% a −8% según recorte, ago subreportado) | `cambio_calidad`: solo `post_2026`. `carnet`: 19m excl. ago, sin separar por régimen, `yearly_seasonality` a evaluar |

Ver decisiones formalizadas en `docs/decisiones_modelado.md`.